## Exploring categorical features

In [ ]:
# Import pandas
import pandas as pd

# Read 'gapminder.csv' into a DataFrame: df
df = pd.read_csv('gapminder.csv')
# Create a boxplot of life expectancy per region  ## Here, 'Region is a categorical feature'
df.boxplot('life', 'Region', rot=60)

# Show the plot
plt.show()

### Creating dummy variables

In [ ]:
scikit-learn does not accept non-numerical features. You saw in the previous exercise that the 'Region' feature 
contains very useful information that can predict life expectancy. For example, Sub-Saharan Africa has a lower life
expectancy compared to Europe and Central Asia. Therefore, if you are trying to predict life expectancy, 
it would be preferable to retain the 'Region' feature. To do this, you need to binarize 
it by creating dummy variables, which is what you will do in this exercise.

In [ ]:
# Create dummy variables: df_region
df_region = pd.get_dummies(df)

# Print the columns of df_region
print(df_region.columns)

# Create dummy variables with drop_first=True: df_region
df_region = pd.get_dummies(df,drop_first=True)

# Print the new columns of df_region
print(df_region.columns)   ## now 'Region America is the dummy variable so it deletes that column from the df .

#### Regression with categorical features

In [ ]:
# Import necessary modules
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

# Instantiate a ridge regressor: ridge
ridge = Ridge(alpha=0.5,normalize=True)

# Perform 5-fold cross-validation: ridge_cv
ridge_cv = cross_val_score(ridge,X,y,cv=5)

# Print the cross-validated scores
print(ridge_cv)

### Dropping missing data

In [ ]:
different datasets encode missing values in different ways. Sometimes it may be a '9999', other times a 0 - real-world
data can be very messy! If you're lucky, the missing values will already be encoded as NaN. We use NaN because it is
an efficient and simplified way of internally representing missing data, and it lets us take advantage of pandas 
methods such as .dropna() and .fillna(), as well as scikit-learn's Imputation transformer Imputer().

In [ ]:
# Convert '?' to NaN
df[df == '?'] = np.nan

# Print the number of NaNs
print(df.isnull().sum())

# Print shape of original DataFrame
print("Shape of Original DataFrame: {}".format(df.shape))

# Drop missing values and print shape of new DataFrame
df = df.dropna()

# Print shape of new DataFrame
print("Shape of DataFrame After Dropping All Rows with Missing Values: {}".format(df.shape))

### Imputing missing data in a ML Pipeline I - With Support Vector Machine ( SVM) Classifier

In [ ]:
As you've come to appreciate, there are many steps to building a model, from creating training and test sets,
to fitting a classifier or regressor, to tuning its parameters, to evaluating its performance on new data. Imputation
can be seen as the first step of this machine learning process, the entirety of which can be viewed within the context
of a pipeline. Scikit-learn provides a pipeline constructor that allows you to piece together these steps into
one process and thereby simplify your workflow.

In [ ]:
# Import the Imputer module
from sklearn.svm import SVC
from sklearn.preprocessing import Imputer

# Setup the Imputation transformer: imp
imp = Imputer(missing_values='NaN', strategy='most_frequent', axis=0)

# Instantiate the SVC classifier: clf
clf = SVC()

# Setup the pipeline with the required steps: steps
 #Create the steps of the pipeline by creating a list of tuples:
   # The first tuple should consist of the imputation step, using imp.
   # The second should consist of the classifier.
steps = [('imputation', imp),
        ('SVM', SVC)]

# NOW WE CAN USE IT TO CLASSIFY

### Imputing missing data in a ML Pipeline II

In [ ]:
What makes pipelines so incredibly useful is the simple interface that they provide. You can use the .fit()
and .predict() methods on pipelines just as you did with your classifiers and regressors!

In [ ]:
The steps of the pipeline have been set up for you, and the feature array X and target variable array y have been
pre-loaded. Additionally, train_test_split and classification_report have been imported from sklearn.model_selection
and sklearn.metrics respectively.

In [ ]:
# Import necessary modules
from sklearn.preprocessing import Imputer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

# Setup the pipeline steps: steps
steps = [('imputation', Imputer(missing_values='NaN', strategy='most_frequent', axis=0)),
        ('SVM', SVC())]

# Create the pipeline: pipeline
pipeline = Pipeline(steps)

# Create training and test sets
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3,random_state=42)

# Fit the pipeline to the train set
pipeline.fit(X_train,y_train)

# Predict the labels of the test set
y_pred = pipeline.predict(X_test)

# Compute metrics
print(classification_report(y_test, y_pred))

### Centering and scaling your data

In [ ]:
the performance of a model can improve if the features are scaled. Note that this is not always the case: In the
Congressional voting records dataset, for example, all of the features are binary. In such a situation, scaling will
have minimal impact.

In [ ]:
# Import scale
from sklearn.preprocessing import scale

# Scale the features: X_scaled
X_scaled = scale(X)

# Print the mean and standard deviation of the unscaled features
print("Mean of Unscaled Features: {}".format(np.mean(X))) 
print("Standard Deviation of Unscaled Features: {}".format(np.std(X)))

# Print the mean and standard deviation of the scaled features
print("Mean of Scaled Features: {}".format(np.mean(X_scaled))) 
print("Standard Deviation of Scaled Features: {}".format(np.std(X_scaled)))

### Centering and scaling in a pipeline and see the performance changes

In [ ]:
# Import the necessary modules
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Setup the pipeline
steps = [('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier())]
        
# Create the pipeline: pipeline
pipeline = Pipeline(steps)

# Create train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Fit the pipeline to the training set: knn_scaled
knn_scaled = pipeline.fit(X_train, y_train)

# Instantiate and fit a k-NN classifier to the unscaled data
knn_unscaled = KNeighborsClassifier().fit(X_train, y_train)

# Compute and print metrics
print('Accuracy with Scaling: {}'.format(knn_scaled.score(X_test, y_test)))
print('Accuracy without Scaling: {}'.format(knn_unscaled.score(X_test, y_test)))

#### Pipeline for classification

In [ ]:
The following modules have been pre-loaded: Pipeline, svm, train_test_split, GridSearchCV, classification_report,
    accuracy_score. The feature and target variable arrays X and y have also been pre-loaded.

In [ ]:
# Setup the pipeline
steps = [('scaler', StandardScaler()),
         ('SVM', SVC())]

pipeline = Pipeline(steps)

# Specify the hyperparameter space , Specify the hyperparameter space using the following notation: 
# 'step_name__parameter_name'. Here, the step_name is SVM, and the parameter_names are C and gamma.
parameters = {'SVM__C':[1, 10, 100],
              'SVM__gamma':[0.1, 0.01]}

# Create train and test sets
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=21)

# Instantiate the GridSearchCV object: cv
cv = GridSearchCV(pipeline,parameters,cv=3)

# Fit to the training set
cv.fit(X_train,y_train)

# Predict the labels of the test set: y_pred
y_pred = cv.predict(X_test)

# Compute and print metrics
print("Accuracy: {}".format(cv.score(X_test, y_test)))
print(classification_report(y_test, y_pred))
print("Tuned Model Parameters: {}".format(cv.best_params_))

### Pipeline for regression

In [ ]:
our job is to build a pipeline that imputes the missing data, scales the features, and fits an ElasticNet to the
Gapminder data. You will then tune the l1_ratio of your ElasticNet using GridSearchCV.All the necessary modules have,
been imported, and the feature and target variable arrays have been pre-loaded as X and y.

In [ ]:
# Setup the pipeline steps: steps
steps = [('imputation', Imputer(missing_values='NaN', strategy='mean', axis=0)),
         ('scaler', StandardScaler()),
         ('elasticnet', ElasticNet())]
         
# Create the pipeline: pipeline
pipeline = Pipeline(steps)

# Specify the hyperparameter space,Specify the hyperparameter space for the l1 ratio using the following notation:
# 'step_name__parameter_name'. Here, the step_name is elasticnet, and the parameter_name is l1_ratio.
parameters = {'elasticnet__l1_ratio':np.linspace(0,1,30)}

# Create train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

# Create the GridSearchCV object: gm_cv
gm_cv = GridSearchCV(pipeline, parameters,cv=3)

# Fit to the training set
gm_cv.fit(X_train, y_train)

# Compute and print the metrics
r2 = gm_cv.score(X_test, y_test)
print("Tuned ElasticNet Alpha: {}".format(gm_cv.best_params_))
print("Tuned ElasticNet R squared: {}".format(r2))